[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/masato-ka/mjlab-handson/blob/main/mjlab_cartpole_handson.ipynb)

# mjlabハンズオン：公式Cartpoleチュートリアル + 改変で学ぶ強化学習の勘どころ

このNotebookは3つの流れで構成されています。

1. **Baseline実装**:わざと「問題のある」Observation/Rewardから出発し、Actions/Events/Terminationsも含めて1つずつ自分で実装しながら、正しいbaseline実装に辿り着く
2. **ドメインランダマイゼーション**:物理パラメータをランダム化して頑健性を高める
3. **コマンド入力の実装**:目標位置を外部から指定できるタスクに拡張する

全部で3本の学習(baseline / domain_rand / command)を回します。理論的な背景(PPOの仕組み、On-policy/Off-policyなど)はCraftドキュメント側にまとめてあるので、このNotebookは実装と実行に集中します。

このNotebookはGoogle Colab想定です。**ランタイム → ランタイムのタイプを変更 → GPU (T4)** を選択してから実行してください。


---
# 0. セットアップ

mjlab(v1.1.0以降、依存関係込みでPyPI公開)と、学習ログ用のWandBをインストールします。


In [ ]:
!nvidia-smi


In [ ]:
# mjlabはベータ版のためAPIが変わることがあります。うまく動かない場合は
# https://github.com/mujocolab/mjlab のREADMEで最新のインストール手順を確認してください。
!pip install -q mjlab
!pip install -q rsl-rl-lib wandb mediapy

import torch
print("CUDA available:", torch.cuda.is_available())
print("Device name   :", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "N/A")

import wandb
wandb.login()  # 初回はAPIキー入力、またはアカウントなしで匿名実行を選択


---
# 1. Baseline実装

ここでは、CartPole(swing-up)のbaseline実装を**Observation → Actions → Events → Rewards → Terminations**の5つの要素に分けて、1つずつ自分の手で組み立てていきます。

- **Observation**と**Rewards**は、あえて「問題のある」実装から出発します。何が問題かを確認したうえで、正しい実装に自分で直してください
- **Actions**・**Events**・**Terminations**は、設計方針を読んで実装する演習です(こちらは最初から間違っているわけではありません)

5つすべてが揃ったら、最後に組み立てて実際に学習を実行します。この流れ自体が、Craftドキュメントの「ベースラインのコード解説」で説明した`ManagerBasedRlEnvCfg`の各プロパティに1対1で対応しています。

## MJCF(cartpole.xml)の準備

まずは物理構造の定義(MJCF)を用意します。この部分に問題はないので、そのまま使います。


In [ ]:
%%writefile cartpole.xml
<mujoco model="cartpole">
  <option timestep="0.01"/>

  <default>
    <default class="pole">
      <joint type="hinge" axis="0 1 0" damping="2e-6"/>
      <geom type="capsule" fromto="0 0 0 0 0 1" size="0.045" material="pole" mass=".1"/>
    </default>
  </default>

  <asset>
    <texture name="grid" type="2d" builtin="checker" rgb1=".1 .2 .3" rgb2=".2 .3 .4" width="300" height="300"/>
    <material name="grid" texture="grid" texrepeat="8 8" reflectance=".2"/>
    <material name="cart" rgba=".7 .1 .1 1"/>
    <material name="pole" rgba=".1 .5 .8 1"/>
    <material name="decoration" rgba=".3 .3 .3 1"/>
  </asset>

  <worldbody>
    <light name="light" pos="0 0 6"/>
    <camera name="fixed" pos="0 -4 1" zaxis="0 -1 0"/>
    <camera name="lookatcart" mode="targetbody" target="cart" pos="0 -2 2"/>
    <geom name="floor" pos="0 0 -.05" size="4 4 .2" type="plane" material="grid"/>
    <geom name="rail1" type="capsule" pos="0 .07 1" zaxis="1 0 0" size="0.02 2" material="decoration"/>
    <geom name="rail2" type="capsule" pos="0 -.07 1" zaxis="1 0 0" size="0.02 2" material="decoration"/>
    <body name="cart" pos="0 0 1">
      <joint name="slider" type="slide" limited="true" axis="1 0 0" range="-1.8 1.8" solreflimit=".08 1" damping="5e-4"/>
      <geom name="cart" type="box" size="0.2 0.15 0.1" material="cart" mass="1"/>
      <body name="pole_1" childclass="pole">
        <joint name="hinge_1"/>
        <geom name="pole_1"/>
      </body>
    </body>
  </worldbody>

  <actuator>
    <motor name="slide" joint="slider" gear="10" ctrllimited="true" ctrlrange="-1 1"/>
  </actuator>
</mujoco>


## Entity・Actuatorの設定(準備)

Observation以降の演習に入る前に、シミュレーション対象そのもの(Entity)とアクチュエータの設定を用意しておきます。ここはCraftドキュメントの「Entityの設定」「アクチュエーターの設定」で説明済みの内容なので、そのまま使います。


In [ ]:
import math
from pathlib import Path
import mujoco
import torch

from mjlab.envs import ManagerBasedRlEnvCfg
from mjlab.envs.mdp import (
    joint_pos_rel,
    joint_vel_rel,
    reset_joints_by_offset,
    time_out,
    JointEffortActionCfg,
)
from mjlab.managers import (
    EventTermCfg,
    ObservationGroupCfg,
    ObservationTermCfg,
    RewardTermCfg,
    SceneEntityCfg,
    TerminationTermCfg,
)
from mjlab.scene import SceneCfg
from mjlab.terrains import TerrainEntityCfg
from mjlab.sim import MujocoCfg, SimulationCfg
from mjlab.entity import EntityCfg, EntityArticulationInfoCfg
from mjlab.actuator import XmlActuatorCfg

_CARTPOLE_XML = Path("cartpole.xml").resolve()

def _get_spec() -> mujoco.MjSpec:
    """MuJoCo モデルスペックを XML から読み込む。"""
    return mujoco.MjSpec.from_file(str(_CARTPOLE_XML))

_CARTPOLE_ARTICULATION = EntityArticulationInfoCfg(
    actuators=(XmlActuatorCfg(target_names_expr=("slider",)),),
)

_BALANCE_INIT = EntityCfg.InitialStateCfg(
    joint_pos={"slider": 0.0, "hinge_1": 0.0},
    joint_vel={".*": 0.0},
)
_SWINGUP_INIT = EntityCfg.InitialStateCfg(
    joint_pos={"slider": 0.0, "hinge_1": math.pi},
    joint_vel={".*": 0.0},
)

def _get_cartpole_entity_cfg(swing_up: bool = True) -> EntityCfg:
    return EntityCfg(
        spec_fn=_get_spec,
        articulation=_CARTPOLE_ARTICULATION,
        init_state=_SWINGUP_INIT if swing_up else _BALANCE_INIT,
    )

# 観測・行動・報酬などを組み立てるための共通のSceneEntityCfg
cart_cfg = SceneEntityCfg("cartpole", joint_names=("slider",))
hinge_cfg = SceneEntityCfg("cartpole", joint_names=("hinge_1",))

print("Entity/Actuatorの準備完了")


## Observationの実装

### 現状(問題のある実装)

まずは次の観測を使ってみます。ポール角度をそのまま(生角度)使う実装です。

```python
observations_broken = {
    "actor": ObservationGroupCfg({
        "cart_pos": ObservationTermCfg(func=joint_pos_rel, params={"asset_cfg": cart_cfg}),
        "pole_angle_raw": ObservationTermCfg(func=joint_pos_rel, params={"asset_cfg": hinge_cfg}),
        "cart_vel": ObservationTermCfg(func=joint_vel_rel, params={"asset_cfg": cart_cfg}),
        "pole_vel": ObservationTermCfg(func=joint_vel_rel, params={"asset_cfg": hinge_cfg}),
    }),
}
```

### 生角度観測は状態に対して一意にならない

今回のswing-upタスクではヒンジに回転制限がありません。生の角度θをそのまま観測に使うと、**見た目が同じ姿勢でも何周したかによってθの値が変わってしまい**、同じ状態に別の観測値が割り当たる非マルコフな観測になります。

### 実装方針

角度をそのまま渡す代わりに、**周期的で連続な表現**であるcos/sinのペアに変換します。

$$
\text{obs}_{\text{angle}} = [\cos\theta,\ \sin\theta]
$$

この表現なら、θが0でも2πでも4πでも(見た目が同じ姿勢なら)まったく同じ観測値になります。また値域が常に[-1, 1]に収まるため、学習の数値的な安定性にも寄与します。

以下のセルの`TODO`を埋めて、`pole_angle_cos_sin`と`observations`ディクショナリを実装してください。**実装例はこのNotebookには載せていません。**詳しい書き方や設計意図はCraftドキュメントの「Observation」「Observationの実装」セクションを参照してください。


In [ ]:
def pole_angle_cos_sin(env, asset_cfg: SceneEntityCfg) -> torch.Tensor:
    """ポール角度をcos/sinのペア([N, 2])で返す。"""
    asset = env.scene[asset_cfg.name]
    angle = asset.data.joint_pos[:, asset_cfg.joint_ids]  # [N, 1]

    # TODO: angle から cos と sin を計算し、torch.cat で [N, 2] にまとめて return してください
    raise NotImplementedError

# TODO: cart_pos / pole_angle(上の関数を使う) / cart_vel / pole_vel の4項目からなる
#       observations ディクショナリ(actor/criticの2グループ)を組み立ててください
observations = None


## Actionsの実装

`actions`は、方策の出力(ポリシーが返す数値)を実際にどのアクチュエータへ、どう変換して渡すかを定義します。今回は「sliderジョイントのモーターに、ポリシーの出力をそのまま(scale=1.0)トルクとして渡す」というシンプルな構成です。

`JointEffortActionCfg`は次の3つを指定します。

| 引数 | 意味 |
| --- | --- |
| `entity_name` | どのEntityに属するアクチュエータか |
| `actuator_names` | 対象アクチュエータの名前(XMLの`<motor name="...">`と対応) |
| `scale` | ポリシー出力に掛けるスケール係数 |

以下のセルの`TODO`を埋めて、`actions`ディクショナリを実装してください。キー名は`"effort"`にしてください。**実装例はこのNotebookには載せていません。**Craftドキュメントの「Actions」セクションを参照してください。


In [ ]:
# TODO: キー名 "effort" に JointEffortActionCfg(entity_name="cartpole", actuator_names=("slider",), scale=1.0)
#       を割り当てた actions ディクショナリを組み立ててください
actions = None


## Eventsの実装

`events`は、シミュレーションのライフサイクル上の特定のタイミングで発火する処理を定義します。今回はエピソードのリセット時(`mode="reset"`)に、カート位置とポール角度を少しだけランダムな初期値にする処理を実装します。これにより、毎回まったく同じ初期状態から始まるのを避け、方策が特定の初期条件に過剰適合するのを防ぎます。

組み込み関数`reset_joints_by_offset`は、対象関節のデフォルト姿勢に対して`position_range`/`velocity_range`で指定した範囲の一様ランダムなオフセットを加えます。

以下のセルの`TODO`を埋めて、`slider`と`hinge_1`それぞれに対する`EventTermCfg`を実装してください(位置レンジは`(-0.1, 0.1)`、速度レンジは`(-0.01, 0.01)`とします)。**実装例はこのNotebookには載せていません。**Craftドキュメントの「Events」セクションを参照してください。


In [ ]:
# TODO: reset_joints_by_offset を使って、"reset_slider" と "reset_hinge" の
#       2つの EventTermCfg(mode="reset") からなる events ディクショナリを組み立ててください
events = None


## Rewardsの実装

### 現状(問題のある実装)

まずは次の報酬を使ってみます。生きている間、毎ステップ一律で+1を返すだけの実装です。

```python
def alive_bonus(env) -> torch.Tensor:
    return torch.ones(env.num_envs, device=env.device)

rewards_broken = {"alive": RewardTermCfg(func=alive_bonus, weight=1.0)}
```

### 一律な報酬は学習信号にならない

この報酬は毎ステップ与えられるという意味では**疎(sparse)ではなく密(dense)な報酬**です。問題は疎密ではなく、**状態や行動によらず常に同じ値を返す**ため、方策の良し悪しをまったく区別できない点にあります。しかも今回のCartPoleは`time_out`以外の終了条件を持たないため、どんな行動を取ってもエピソードの長さは変わらず、結果としてこの報酬はどんな方策に対しても完全に同じ累積報酬を返します。つまりこれは、密ではあるが**学習信号としてまったく機能しない報酬**です。

### 実装方針

4つの要素をすべて**0〜1に正規化して掛け合わせる**設計にします。どれか1つでも0に近づくと報酬全体が下がるため、「全部同時に満たす」ことを学習させられます。

$$
r = \text{upright} \times \text{centered} \times \text{small\_ctrl} \times \text{small\_vel}
$$

| 要素 | 意味 | 実装方針 |
|---|---|---|
| upright | ポールが直立(θ=0)に近いほど1 | $(\cos\theta + 1) / 2$ |
| centered | カートが中央(x=0)に近いほど1 | ガウス型のtolerance関数を使い、$(1 + \text{tolerance}(x,\ \text{margin}=1.0)) / 2$ |
| small_ctrl | 操作量が小さいほど1 | $(4 + \text{tolerance}(u,\ \text{margin}=1.0)) / 5$ |
| small_vel | ポール角速度が小さいほど1 | $(1 + \text{tolerance}(\dot\theta,\ \text{margin}=5.0)) / 2$ |

`tolerance(x, margin)`は「|x|がmargin以下でおよそ1、marginから離れるほど0に近づく」関数です。ガウス関数 $\exp(-0.5 (x/\text{margin})^2)$ で実装できます。

以下のセルの`TODO`部分を埋めて、`_tolerance`・`cartpole_smooth_reward`・`rewards`ディクショナリを実装してください。**実装例はこのNotebookには載せていません。**Craftドキュメントの「Reward」「cartpole_smooth_rewardの実装」セクションを参照してください。


In [ ]:
def _tolerance(x: torch.Tensor, margin: float) -> torch.Tensor:
    """|x|がmargin以下で1に近く、marginから離れるほど0に近づくガウス型関数。
    ヒント: exp(-0.5 * (x / margin) ** 2)
    """
    # TODO: ここを実装してください
    raise NotImplementedError

def cartpole_smooth_reward(env, cart_cfg: SceneEntityCfg, hinge_cfg: SceneEntityCfg) -> torch.Tensor:
    cart = env.scene[cart_cfg.name]
    angle = cart.data.joint_pos[:, hinge_cfg.joint_ids]
    cos_angle = torch.cos(angle).squeeze(-1)
    cart_pos = cart.data.joint_pos[:, cart_cfg.joint_ids].squeeze(-1)
    pole_vel = cart.data.joint_vel[:, hinge_cfg.joint_ids].squeeze(-1)
    ctrl = env.action_manager.action.squeeze(-1)

    # TODO: 上の表の4つの要素(upright, centered, small_ctrl, small_vel)を実装し、
    #       それらを掛け合わせて return してください
    raise NotImplementedError

# TODO: 上の cartpole_smooth_reward を使って rewards ディクショナリを組み立ててください
rewards = None


## Terminationsの実装

`terminations`は、いつエピソードを終えるかを定義します。今回のswing-upタスクでは、ポールが倒れても物理的にエピソードを打ち切る条件はなく、`episode_length_s`で指定した時間が経過したら終了する`time_out`のみを使います。「倒れた状態」もタスクの一部(そこから立て直す)として扱われるためです。

`TerminationTermCfg(time_out=True)`のフラグは、この終了が「失敗による終了」ではなく「時間切れによる打ち切り(truncation)」であることをトレーニングフレームワークに伝えるためのものです。この区別により、価値関数の学習時に時間切れの状態を「そこで報酬がゼロになる終端」として扱わず、続きの価値を推定(bootstrap)できるようになります。

以下のセルの`TODO`を埋めて、`terminations`ディクショナリを実装してください。**実装例はこのNotebookには載せていません。**Craftドキュメントの「terminations」セクションを参照してください。


In [ ]:
# TODO: 組み込み関数 time_out を使い、time_out=True を指定した TerminationTermCfg で
#       terminations ディクショナリを組み立ててください
terminations = None


## 組み立て:ManagerBasedRlEnvCfg

Observation・Actions・Events・Rewards・Terminationsが揃ったので、`ManagerBasedRlEnvCfg`として1つにまとめます。


In [ ]:
def cartpole_env_cfg(swing_up: bool = True, num_envs: int = 512) -> ManagerBasedRlEnvCfg:
    """Cartpole環境の完全な設定を返す(Observation/Actions/Events/Rewards/Terminationsを組み立てる)。"""
    return ManagerBasedRlEnvCfg(
        scene=SceneCfg(
            terrain=TerrainEntityCfg(terrain_type="plane"),
            entities={"cartpole": _get_cartpole_entity_cfg(swing_up=swing_up)},
            num_envs=num_envs,
            env_spacing=4.0,
        ),
        observations=observations,
        actions=actions,
        events=events,
        rewards=rewards,
        terminations=terminations,
        sim=SimulationCfg(mujoco=MujocoCfg(timestep=0.01, disableflags=("contact",))),
        decimation=5,
        episode_length_s=50.0,
    )

print("cartpole_env_cfg 組み立て完了")


## PPOの設定と学習の実行

環境側の実装が完了したので、PPOの設定(actor/critic構造とハイパーパラメータ)を作り、学習を実行します。ハイパーパラメータの意味(`clip_param`・`entropy_coef`・`gamma`/`lam`など)はCraftドキュメントの「PPOのポリシー設定」で詳しく説明しているので、ここではそのまま使います。


In [ ]:
from mjlab.rl import RslRlOnPolicyRunnerCfg, RslRlModelCfg, RslRlPpoAlgorithmCfg

def cartpole_ppo_runner_cfg(entropy_coef: float = 0.005) -> RslRlOnPolicyRunnerCfg:
    return RslRlOnPolicyRunnerCfg(
        actor=RslRlModelCfg(
            class_name="MLPModel", hidden_dims=(64, 64), activation="elu",
            distribution_cfg={"class_name": "GaussianDistribution", "init_std": 1.0, "std_type": "scalar"},
        ),
        critic=RslRlModelCfg(
            class_name="MLPModel", hidden_dims=(64, 64), activation="elu", distribution_cfg=None,
        ),
        algorithm=RslRlPpoAlgorithmCfg(
            value_loss_coef=1.0, use_clipped_value_loss=True, clip_param=0.2,
            entropy_coef=entropy_coef, num_learning_epochs=5, num_mini_batches=4,
            learning_rate=1e-3, schedule="adaptive", gamma=0.99, lam=0.95,
            desired_kl=0.01, max_grad_norm=1.0,
        ),
        num_steps_per_env=24,
        max_iterations=500,
        save_interval=100,
        experiment_name="cartpole_handson",
        run_name="baseline",
        logger="wandb",
    )

print("cartpole_ppo_runner_cfg 定義完了")


In [ ]:
import dataclasses
from pathlib import Path

from mjlab.envs import ManagerBasedRlEnv
from mjlab.rl import MjlabOnPolicyRunner, RslRlVecEnvWrapper

device = "cuda:0"

def run_training(env_cfg, rl_cfg, run_name: str, max_iters: int = 500):
    """env_cfgとrl_cfgを受け取り学習を実行する共通ヘルパー。
    以降の実験すべてでこの関数を使い回す。
    """
    rl_cfg.run_name = run_name
    rl_cfg.max_iterations = max_iters

    env = ManagerBasedRlEnv(cfg=env_cfg, device=device)
    vec_env = RslRlVecEnvWrapper(env)

    log_dir = str(Path("logs") / rl_cfg.experiment_name / run_name)
    runner = MjlabOnPolicyRunner(
        env=vec_env,
        train_cfg=dataclasses.asdict(rl_cfg),
        log_dir=log_dir,
        device=device,
    )
    print(f"[train] run={run_name} を開始します (ログ: {log_dir})")
    runner.learn(num_learning_iterations=rl_cfg.max_iterations)
    env.close()
    print(f"[train] run={run_name} 完了")
    return log_dir

print("run_training ヘルパー定義完了")


### 学習を実行

自分で実装したObservation/Actions/Events/Rewards/Terminationsが揃ったbaseline実装で、実際に学習します。WandBの実験名`cartpole_handson`・run名`baseline`として記録され、以降の実験(ドメインランダマイゼーション・コマンド入力)と同じダッシュボードで比較できます。


In [ ]:
NUM_ENVS = 512
MAX_ITERS = 500  # 時間がなければ200程度に減らしても傾向は見える

baseline_env_cfg = cartpole_env_cfg(swing_up=True, num_envs=NUM_ENVS)
baseline_rl_cfg = cartpole_ppo_runner_cfg(entropy_coef=0.005)

log_dir_baseline = run_training(baseline_env_cfg, baseline_rl_cfg, run_name="baseline", max_iters=MAX_ITERS)


---
## 学習済みbaselineポリシーの動画デモ(Colab上)

ローカル環境の準備をする前に、まずはColab上だけでbaselineの学習済みポリシーを動かしてみます。ディスプレイは使わず、`mediapy`でオフスクリーン動画として表示するので、ローカル環境は不要です。

In [ ]:
import glob
import numpy as np
import mediapy as media
from rsl_rl.models.mlp_model import MLPModel

ckpts = sorted(glob.glob(f"{log_dir_baseline}/model_*.pt"))
assert ckpts, "チェックポイントが見つかりません。baselineの学習セルを先に実行してください。"
checkpoint_path = ckpts[-1]
print("使用するチェックポイント:", checkpoint_path)

play_cfg = cartpole_env_cfg(swing_up=True, num_envs=1)
play_cfg.episode_length_s = 9999.0
play_env = ManagerBasedRlEnv(cfg=play_cfg, device=device)

obs_dim = play_env.observation_space.spaces["actor"].shape[-1]
act_dim = play_env.action_space.shape[-1]

dummy_obs = {"actor": torch.zeros(1, obs_dim, device=device)}
actor = MLPModel(
    obs=dummy_obs, obs_groups={"actor": ["actor"]}, obs_set="actor",
    output_dim=act_dim, hidden_dims=(64, 64), activation="elu",
    distribution_cfg={
        "class_name": "rsl_rl.modules.distribution.GaussianDistribution",
        "init_std": 1.0, "std_type": "scalar",
    },
).to(device)

checkpoint = torch.load(checkpoint_path, map_location=device)
actor.load_state_dict(checkpoint["actor_state_dict"])
actor.eval()

frames = []
obs_dict, _ = play_env.reset()
n_steps = 600
for step in range(n_steps):
    with torch.no_grad():
        action = actor(obs_dict)
    obs_dict, reward, terminated, truncated, info = play_env.step(action)

    frame = play_env.render()
    if frame is not None:
        frames.append(frame)

    if terminated.any() or truncated.any():
        obs_dict, _ = play_env.reset()

print(f"{len(frames)} フレームを収集しました")
if frames:
    fps = int(1 / (play_cfg.sim.mujoco.timestep * play_env.cfg.decimation))
    media.show_video(frames, fps=fps)
else:
    print("render()がフレームを返しませんでした。mjlabのバージョンによってrender APIが異なる場合があるため、"
          "mjlab.envs.ManagerBasedRlEnv のドキュメントで最新のオフスクリーン描画方法を確認してください。")

---
## 学習済みbaselineポリシーを動かす

baselineの学習が終わったら、次の章に進む前にこのポリシーを実際に動かして確認します。ここで用意する2通りのローカル実行方法は、この後のTier2a・Tier3でもチェックポイント/ONNXファイルを差し替えるだけでそのまま使い回します。

1. **ローカルGPUでのmjlab実行**(`play_local.py`):キーボード操作やビューア標準機能での外乱付与など、対話的に確認できる
2. **CPUのみでのONNX実行**(`infer_local.py`):PyTorch・mjlab・GPUのいずれも不要で、GPUを持たないマシンでも動かせる

GPUを持たないマシンで作業している場合は、2のONNX実行だけで動作確認できます。

In [ ]:
from google.colab import files
import glob

baseline_ckpts = sorted(glob.glob(f"{log_dir_baseline}/model_*.pt"))
checkpoint_path_baseline = baseline_ckpts[-1]
print("使用するチェックポイント:", checkpoint_path_baseline)

files.download(checkpoint_path_baseline)
files.download("cartpole.xml")

### ローカル環境の準備

1. NVIDIA GPU搭載のマシン(学習も試したい場合)、または**評価のみ**ならmacOSでも可
2. [uv](https://docs.astral.sh/uv/)をインストール(公式サイトの手順に従ってください: https://docs.astral.sh/uv/getting-started/installation/)
3. `play_local.py`の先頭にmjlab・rsl-rl-libを依存として宣言したインラインスクリプトメタデータ(PEP 723)を埋め込んであるので、`pip install`は不要です
4. 直前のセルでダウンロードした`cartpole.xml`と`model_*.pt`(チェックポイント)を、次のセルで生成する`play_local.py`と同じフォルダに置く

### 実行方法

```bash
uv run play_local.py --checkpoint model_499.pt --task baseline
```

この`play_local.py`は、この後のTier2a・Tier3でもそのまま使い回します(`--task`とチェックポイントを差し替えるだけです)。

| `--task`の値 | 対応するチェックポイント | 学習する章 |
|---|---|---|
| `baseline` | baseline | 1章(ここ) |
| `tier2a_domain_rand` | tier2a_domain_rand | 2章 |
| `tier3` | tier3_command_conditioned | 3章 |

観測・行動の次元が合わないチェックポイントを誤ったタスクで読み込むと、`load_state_dict`が入力層のサイズ不一致でエラーを出します。

### 操作方法

- **Ctrl+右クリックドラッグ**(MuJoCoビューア標準機能):カートやポールに直接外力を加えて崩し、立て直せるか試す
- **Qキー、またはウィンドウを閉じる**:終了

(← / →キーによる目標カート位置の操作は、コマンド入力を学習するTier3でのみ有効です)

> **注意**:`NativeMujocoViewer`のループの回し方(`tick()`の呼び方や終了判定)はmjlabのバージョンで変わることがあります。下記スクリプトで動かない場合は、[mjlab.viewerのAPIドキュメント](https://mujocolab.github.io/mjlab/main/source/api/viewer.html)や、公式の`play`スクリプト(`uv run play <task> --viewer native`)の実装を参考に調整してください。

In [ ]:
%%writefile play_local.py
#!/usr/bin/env -S uv run
# /// script
# requires-python = ">=3.10"
# dependencies = [
#     "mjlab",
#     "rsl-rl-lib",
# ]
# ///
"""
play_local.py
==============
学習済みcartpole方策を、MuJoCoのネイティブビューアでインタラクティブに実行するスクリプト。
Google Colabでは実行できません(ディスプレイとリアルタイムのキー入力が必要なため)。
ディスプレイのあるローカルPC(NVIDIA GPU、または評価のみならmacOSでも可)で実行してください。

事前準備:
  uvをインストールしておく(公式サイト参照: https://docs.astral.sh/uv/getting-started/installation/)。
  パッケージのインストールは不要。上のインラインスクリプトメタデータ(PEP 723)を
  uv run が読み取り、実行時に自動で一時的な仮想環境を構築する。
  このファイルと同じフォルダに cartpole.xml とチェックポイント(.pt)を置く

実行例:
  uv run play_local.py --checkpoint model_499.pt --task baseline
  uv run play_local.py --checkpoint model_499.pt --task tier2a_domain_rand
  uv run play_local.py --checkpoint model_499.pt --task tier3

操作方法:
  ← / → キー         : (tier3のみ)目標カート位置を左右に動かす
  Ctrl+右クリックドラッグ : カート/ポールに外力を加える(ビューア標準機能。外乱耐性の確認に使える)
  Q / ウィンドウを閉じる : 終了
"""
import argparse
import math
from pathlib import Path

import mujoco
import torch

from mjlab.entity import EntityCfg, EntityArticulationInfoCfg
from mjlab.actuator import XmlActuatorCfg
from mjlab.envs import ManagerBasedRlEnvCfg, ManagerBasedRlEnv
from mjlab.envs.mdp import (
    joint_pos_rel, joint_vel_rel, reset_joints_by_offset, time_out,
    JointEffortActionCfg, generated_commands,
)
from mjlab.managers import (
    EventTermCfg, ObservationGroupCfg, ObservationTermCfg,
    RewardTermCfg, SceneEntityCfg, TerminationTermCfg,
)
from mjlab.managers.command_manager import CommandTerm, CommandTermCfg
from mjlab.scene import SceneCfg
from mjlab.terrains import TerrainEntityCfg
from mjlab.sim import MujocoCfg, SimulationCfg
from mjlab.viewer import NativeMujocoViewer

_CARTPOLE_XML = Path(__file__).parent / "cartpole.xml"


def _get_spec() -> mujoco.MjSpec:
    return mujoco.MjSpec.from_file(str(_CARTPOLE_XML))


_ARTICULATION = EntityArticulationInfoCfg(actuators=(XmlActuatorCfg(target_names_expr=("slider",)),))
_SWINGUP_INIT = EntityCfg.InitialStateCfg(joint_pos={"slider": 0.0, "hinge_1": math.pi}, joint_vel={".*": 0.0})


def _entity_cfg() -> EntityCfg:
    return EntityCfg(spec_fn=_get_spec, articulation=_ARTICULATION, init_state=_SWINGUP_INIT)


def pole_angle_cos_sin(env, asset_cfg: SceneEntityCfg) -> torch.Tensor:
    asset = env.scene[asset_cfg.name]
    angle = asset.data.joint_pos[:, asset_cfg.joint_ids]
    return torch.cat([torch.cos(angle), torch.sin(angle)], dim=-1)


def _tolerance(x: torch.Tensor, margin: float) -> torch.Tensor:
    return torch.exp(-0.5 * (x / margin) ** 2)


def cartpole_smooth_reward(env, cart_cfg: SceneEntityCfg, hinge_cfg: SceneEntityCfg) -> torch.Tensor:
    cart = env.scene[cart_cfg.name]
    angle = cart.data.joint_pos[:, hinge_cfg.joint_ids]
    cos_angle = torch.cos(angle).squeeze(-1)
    cart_pos = cart.data.joint_pos[:, cart_cfg.joint_ids].squeeze(-1)
    pole_vel = cart.data.joint_vel[:, hinge_cfg.joint_ids].squeeze(-1)
    ctrl = env.action_manager.action.squeeze(-1)
    upright = (cos_angle + 1.0) / 2.0
    centered = (1.0 + _tolerance(cart_pos, margin=1.0)) / 2.0
    small_ctrl = (4.0 + _tolerance(ctrl, margin=1.0)) / 5.0
    small_vel = (1.0 + _tolerance(pole_vel, margin=5.0)) / 2.0
    return upright * centered * small_ctrl * small_vel


def baseline_env_cfg(num_envs: int = 1) -> ManagerBasedRlEnvCfg:
    cart_cfg = SceneEntityCfg("cartpole", joint_names=("slider",))
    hinge_cfg = SceneEntityCfg("cartpole", joint_names=("hinge_1",))
    actor_terms = {
        "cart_pos": ObservationTermCfg(func=joint_pos_rel, params={"asset_cfg": cart_cfg}),
        "pole_angle": ObservationTermCfg(func=pole_angle_cos_sin, params={"asset_cfg": hinge_cfg}),
        "cart_vel": ObservationTermCfg(func=joint_vel_rel, params={"asset_cfg": cart_cfg}),
        "pole_vel": ObservationTermCfg(func=joint_vel_rel, params={"asset_cfg": hinge_cfg}),
    }
    return ManagerBasedRlEnvCfg(
        scene=SceneCfg(
            terrain=TerrainEntityCfg(terrain_type="plane"),
            entities={"cartpole": _entity_cfg()}, num_envs=num_envs, env_spacing=4.0,
        ),
        observations={"actor": ObservationGroupCfg(actor_terms), "critic": ObservationGroupCfg({**actor_terms})},
        actions={"effort": JointEffortActionCfg(entity_name="cartpole", actuator_names=("slider",), scale=1.0)},
        events={
            "reset_slider": EventTermCfg(func=reset_joints_by_offset, mode="reset",
                params={"position_range": (-0.1, 0.1), "velocity_range": (-0.01, 0.01), "asset_cfg": cart_cfg}),
            "reset_hinge": EventTermCfg(func=reset_joints_by_offset, mode="reset",
                params={"position_range": (-0.1, 0.1), "velocity_range": (-0.01, 0.01), "asset_cfg": hinge_cfg}),
        },
        rewards={"smooth_reward": RewardTermCfg(func=cartpole_smooth_reward, weight=1.0,
                params={"cart_cfg": cart_cfg, "hinge_cfg": hinge_cfg})},
        terminations={"time_out": TerminationTermCfg(func=time_out, time_out=True)},
        sim=SimulationCfg(mujoco=MujocoCfg(timestep=0.01, disableflags=("contact",))),
        decimation=5,
        episode_length_s=9999.0,  # play時はタイムアウトなし
    )


class CartTargetCommand(CommandTerm):
    def __init__(self, cfg: "CartTargetCommandCfg", env):
        super().__init__(cfg, env)
        self._target = torch.zeros(env.num_envs, device=env.device)
        self._external_override = None
        self._cart_joint_idx = list(env.scene["cartpole"].joint_names).index("slider")

    @property
    def command(self) -> torch.Tensor:
        return self._target.unsqueeze(-1)

    def _resample_command(self, env_ids: torch.Tensor):
        if self._external_override is not None:
            self._target[env_ids] = self._external_override[env_ids]
        else:
            self._target[env_ids] = torch.empty(len(env_ids), device=self.device).uniform_(*self.cfg.target_range)

    def _update_command(self, env_ids):
        if self._external_override is not None:
            self._target[:] = self._external_override

    def _update_metrics(self):
        cart = self._env.scene["cartpole"]
        cart_pos = cart.data.joint_pos[:, self._cart_joint_idx]
        self.metrics["cart_target_error"] = (self._target - cart_pos).abs()

    def set_external_target(self, target):
        self._external_override = target


class CartTargetCommandCfg(CommandTermCfg):
    entity_name: str = "cartpole"
    resampling_time_range: tuple = (3.0, 6.0)
    target_range: tuple = (-1.2, 1.2)

    def build(self, env) -> CartTargetCommand:
        return CartTargetCommand(self, env)


def cartpole_command_reward(env, cart_cfg: SceneEntityCfg, hinge_cfg: SceneEntityCfg,
                             command_name: str = "cart_target") -> torch.Tensor:
    cart = env.scene[cart_cfg.name]
    angle = cart.data.joint_pos[:, hinge_cfg.joint_ids]
    cos_angle = torch.cos(angle).squeeze(-1)
    cart_pos = cart.data.joint_pos[:, cart_cfg.joint_ids].squeeze(-1)
    pole_vel = cart.data.joint_vel[:, hinge_cfg.joint_ids].squeeze(-1)
    ctrl = env.action_manager.action.squeeze(-1)
    target = env.command_manager.get_command(command_name).squeeze(-1)
    upright = (cos_angle + 1.0) / 2.0
    centered = (1.0 + _tolerance(cart_pos - target, margin=1.0)) / 2.0
    small_ctrl = (4.0 + _tolerance(ctrl, margin=1.0)) / 5.0
    small_vel = (1.0 + _tolerance(pole_vel, margin=5.0)) / 2.0
    return upright * centered * small_ctrl * small_vel


def command_env_cfg(num_envs: int = 1) -> ManagerBasedRlEnvCfg:
    cfg = baseline_env_cfg(num_envs=num_envs)
    cart_cfg = SceneEntityCfg("cartpole", joint_names=("slider",))
    hinge_cfg = SceneEntityCfg("cartpole", joint_names=("hinge_1",))
    cfg.commands = {"cart_target": CartTargetCommandCfg()}
    actor_terms = {
        "cart_pos": ObservationTermCfg(func=joint_pos_rel, params={"asset_cfg": cart_cfg}),
        "pole_angle": ObservationTermCfg(func=pole_angle_cos_sin, params={"asset_cfg": hinge_cfg}),
        "cart_vel": ObservationTermCfg(func=joint_vel_rel, params={"asset_cfg": cart_cfg}),
        "pole_vel": ObservationTermCfg(func=joint_vel_rel, params={"asset_cfg": hinge_cfg}),
        "cart_target": ObservationTermCfg(func=generated_commands, params={"command_name": "cart_target"}),
    }
    cfg.observations = {"actor": ObservationGroupCfg(actor_terms), "critic": ObservationGroupCfg({**actor_terms})}
    cfg.rewards = {"command_reward": RewardTermCfg(func=cartpole_command_reward, weight=1.0,
                    params={"cart_cfg": cart_cfg, "hinge_cfg": hinge_cfg, "command_name": "cart_target"})}
    return cfg


# ── Tier2-A: ドメインランダム化(観測/行動は共通。イベントも込みにして頑健性を見る) ──
def domain_rand_env_cfg(num_envs: int = 1) -> ManagerBasedRlEnvCfg:
    from mjlab.envs.mdp import dr
    cfg = baseline_env_cfg(num_envs=num_envs)
    pole_cfg = SceneEntityCfg("cartpole", body_names=["pole_1"])
    hinge_j_cfg = SceneEntityCfg("cartpole", joint_names=["hinge_1"])
    slider_j_cfg = SceneEntityCfg("cartpole", joint_names=["slider"])
    cfg.events = dict(cfg.events)
    cfg.events["dr_pole_mass_inertia"] = EventTermCfg(
        mode="reset", func=dr.pseudo_inertia,
        params={"asset_cfg": pole_cfg, "alpha_range": (-0.3, 0.3)},
    )
    cfg.events["dr_hinge_damping"] = EventTermCfg(
        mode="reset", func=dr.joint_damping,
        params={"asset_cfg": hinge_j_cfg, "ranges": (0.5, 3.0), "operation": "scale"},
    )
    cfg.events["dr_slider_damping"] = EventTermCfg(
        mode="reset", func=dr.joint_damping,
        params={"asset_cfg": slider_j_cfg, "ranges": (0.5, 2.0), "operation": "scale"},
    )
    return cfg


# ここに他のTierを追加したい場合は、Notebookの対応する make_*_cfg 関数をコピーして
# この辞書に登録すれば --task で指定できるようになる
TASKS = {
    "baseline": baseline_env_cfg,
    "tier2a_domain_rand": domain_rand_env_cfg,
    "tier3": command_env_cfg,
}


def load_policy(checkpoint_path: str, obs_dim: int, act_dim: int, device: str):
    from rsl_rl.models.mlp_model import MLPModel
    dummy_obs = {"actor": torch.zeros(1, obs_dim, device=device)}
    actor = MLPModel(
        obs=dummy_obs, obs_groups={"actor": ["actor"]}, obs_set="actor",
        output_dim=act_dim, hidden_dims=(64, 64), activation="elu",
        distribution_cfg={"class_name": "rsl_rl.modules.distribution.GaussianDistribution",
                           "init_std": 1.0, "std_type": "scalar"},
    ).to(device)
    checkpoint = torch.load(checkpoint_path, map_location=device)
    actor.load_state_dict(checkpoint["actor_state_dict"])
    actor.eval()
    return actor


class PolicyWrapper:
    """NativeMujocoViewer が要求する policy(obs) -> action の形にラップする。"""

    def __init__(self, actor):
        self.actor = actor

    def __call__(self, obs):
        with torch.no_grad():
            return self.actor({"actor": obs})


def main():
    parser = argparse.ArgumentParser(description="Cartpole mjlab local interactive play")
    parser.add_argument("--checkpoint", required=True, help="学習済みモデル(.pt)のパス")
    parser.add_argument("--task", choices=list(TASKS.keys()), default="tier3")
    parser.add_argument("--gpu-id", type=int, default=0, help="-1でCPU")
    args = parser.parse_args()

    device = f"cuda:{args.gpu_id}" if args.gpu_id >= 0 and torch.cuda.is_available() else "cpu"
    print(f"[play_local] task={args.task} device={device}")

    env_cfg = TASKS[args.task](num_envs=1)
    env = ManagerBasedRlEnv(cfg=env_cfg, device=device)

    obs_dim = env.observation_space.spaces["actor"].shape[-1]
    act_dim = env.action_space.shape[-1]
    actor = load_policy(args.checkpoint, obs_dim, act_dim, device)
    policy = PolicyWrapper(actor)

    has_command = "cart_target" in getattr(env.command_manager, "active_terms", {})
    command_term = env.command_manager.get_term("cart_target") if has_command else None
    target_value = {"v": 0.0}

    def key_callback(keycode: int):
        """GLFWキーコード: 262=Right, 263=Left。← / → で目標カート位置を操作する。"""
        if command_term is None:
            return
        if keycode == 262:
            target_value["v"] = min(target_value["v"] + 0.1, 1.2)
        elif keycode == 263:
            target_value["v"] = max(target_value["v"] - 0.1, -1.2)
        else:
            return
        command_term.set_external_target(torch.full((1,), target_value["v"], device=device))
        print(f"[key] 目標カート位置: {target_value['v']:.2f}")

    viewer = NativeMujocoViewer(
        env=env,
        policy=policy,
        key_callback=key_callback,
        enable_perturbations=True,  # Ctrl+右クリックドラッグで外力を加えられる(標準機能)
    )

    print("ビューアを起動します。ウィンドウを閉じるかQキーで終了します。")
    if hasattr(viewer, "setup"):
        viewer.setup()
    try:
        running = True
        while running:
            running = viewer.tick()
    except KeyboardInterrupt:
        pass
    finally:
        if hasattr(viewer, "close"):
            viewer.close()


if __name__ == "__main__":
    main()


## CPUのみでのONNX実行

上の`play_local.py`は、ローカル側にも**mjlab一式(PyTorch・mujoco-warp・GPU)**が必要でした。ここでは [pollen-robotics/microduck_rl](https://github.com/pollen-robotics/microduck_rl) の `export.py` / `infer_policy.py` と同じ考え方——**学習はGPU上のmjlabで行い、デプロイはCPU上のONNX + 素のmujocoで行う**——を、CartPoleの規模に合わせて最小構成で再現します。

mjlabのランナーは`export_policy_to_onnx()`という組み込みメソッドを持っています。観測の正規化(`EmpiricalNormalization`)を使っている場合、これも自動的に`actor(normalizer(obs))`としてグラフに焼き込まれるため、**手動でONNXに変換する経路は使わない**のがmicroduck_rlのexport.pyでも徹底されているポイントです(今回のCartPole baselineは正規化を使っていませんが、他のタスクに応用する際のために同じ経路を使っておきます)。

In [ ]:
import dataclasses
from pathlib import Path

from mjlab.rl.exporter_utils import get_base_metadata, attach_metadata_to_onnx


def export_to_onnx(env_cfg, rl_cfg, checkpoint_path: str, onnx_path: str, device: str = "cpu") -> Path:
    """学習済みcheckpointをONNXにエクスポートする。

    エクスポート自体は推論グラフをトレースするだけなのでCPUで十分。
    num_envs=1のenv_cfgを渡すこと(バッチ次元1でトレースする)。この後のTier2a・Tier3でも
    env_cfgとcheckpoint_pathを差し替えるだけでそのまま使い回す。
    """
    env = ManagerBasedRlEnv(cfg=env_cfg, device=device)
    vec_env = RslRlVecEnvWrapper(env)

    # 学習時と同じrunnerクラスでcheckpointを読み込む(ネットワーク構造を合わせるため)
    runner = MjlabOnPolicyRunner(
        env=vec_env,
        train_cfg=dataclasses.asdict(rl_cfg),
        log_dir="export_scratch",  # エクスポート時は使い捨てのログ置き場でよい
        device=device,
    )
    runner.load(checkpoint_path, map_location=device)

    onnx_path = Path(onnx_path).resolve()
    runner.export_policy_to_onnx(str(onnx_path.parent), onnx_path.name)

    # 観測/行動の次元などのメタデータをONNXファイル自体に埋め込んでおく(任意だが後で役立つ)
    metadata = get_base_metadata(runner.env.unwrapped)
    attach_metadata_to_onnx(str(onnx_path), metadata)

    env.close()
    print(f"Exported: {onnx_path}")
    return onnx_path


# baselineの学習済みモデルをエクスポート
export_to_onnx(
    env_cfg=cartpole_env_cfg(swing_up=True, num_envs=1),
    rl_cfg=cartpole_ppo_runner_cfg(entropy_coef=0.005),
    checkpoint_path=baseline_ckpts[-1],
    onnx_path="baseline_policy.onnx",
)
files.download("baseline_policy.onnx")

### 事前準備

`play_local.py`と違い、**PyTorchもmjlabも不要**です。Python環境の構築・パッケージ管理・実行はすべて[uv](https://docs.astral.sh/uv/)を使います。

uv自体のインストールは公式サイトの手順に従ってください: https://docs.astral.sh/uv/getting-started/installation/

`infer_local.py`の先頭には必要なパッケージ(`mujoco` / `onnxruntime` / `numpy`)を宣言したインラインスクリプトメタデータ([PEP 723](https://peps.python.org/pep-0723/))を埋め込んであります。そのため`pip install`のような手動でのパッケージインストールは不要で、`uv run`が実行時に自動で一時的な仮想環境を作ってくれます。

先ほどダウンロードした`baseline_policy.onnx`と`cartpole.xml`を、次のセルで生成する`infer_local.py`と同じフォルダに置いてください。

### 実行方法

```bash
uv run infer_local.py --onnx baseline_policy.onnx --task baseline
```

この`infer_local.py`もTier2a・Tier3でそのまま使い回します。Tier2a(`tier2a_domain_rand`)はbaselineと観測形状が同じなので`--task baseline`のまま、Tier3のみ`--task tier3`を指定します。

初回実行時、`uv`が依存パッケージを解決してキャッシュするため少し時間がかかりますが、2回目以降は高速に起動します。

### 操作方法

- **← / → キー**:(`--task tier3`のときのみ)目標カート位置を左右に動かす
- **Ctrl+右クリックドラッグ**:カート/ポールに外力を加える(mujocoビューア標準機能)
- **ウィンドウを閉じる**:終了

> **macOSの注意**:`mujoco.viewer.launch_passive`はmacOSでは通常の`python`コマンドでは動作せず、mujocoパッケージに同梱されている`mjpython`コマンドを代わりに使う必要があります。uv経由で実行する場合は、依存パッケージを明示しつつ`mjpython`をターゲットに指定します。
>
> ```bash
> uv run --with mujoco --with onnxruntime --with numpy mjpython infer_local.py --onnx baseline_policy.onnx --task baseline
> ```
>
> (この`--with`の組み合わせが期待通りに動かない場合は、[uvのドキュメント](https://docs.astral.sh/uv/guides/scripts/)でスクリプト実行の最新の作法を確認してください。)

In [ ]:
%%writefile infer_local.py
#!/usr/bin/env -S uv run
# /// script
# requires-python = ">=3.10"
# dependencies = [
#     "mujoco",
#     "onnxruntime",
#     "numpy",
# ]
# ///
"""
infer_local.py
================
学習済みcartpole方策をONNX(onnxruntime)で読み込み、通常のmujocoパッケージだけで
CPU上で動かすスクリプト。mjlab / PyTorch / GPUは一切不要。

pollen-robotics/microduck_rl の export.py / infer_policy.py と同じ考え方
(学習はGPU上のmjlabで行い、デプロイはCPU上のONNX + mujocoで行う)を、
CartPoleの規模に合わせて最小構成にしたもの。

事前準備:
  uvをインストールしておく(公式サイト参照: https://docs.astral.sh/uv/getting-started/installation/)。
  パッケージのインストールは不要。上のインラインスクリプトメタデータ(PEP 723)を
  uv run が読み取り、実行時に自動で一時的な仮想環境を構築する。
  このファイルと同じフォルダに cartpole.xml と *.onnx を置く

実行例:
  uv run infer_local.py --onnx baseline_policy.onnx --task baseline
  uv run infer_local.py --onnx tier3_policy.onnx --task tier3

操作方法(--task tier3 のときのみ):
  ← / → キー            : 目標カート位置を左右に動かす
  Ctrl+右クリックドラッグ : カート/ポールに外力を加える(mujocoビューア標準機能)
  ウィンドウを閉じる      : 終了

macOSの注意:
  mujoco.viewer.launch_passive は通常のpython/uv runでは動作しません。
  mujocoパッケージに同梱されている mjpython コマンドを使う必要があります
  (例: uv run --with mujoco --with onnxruntime --with numpy mjpython infer_local.py
        --onnx baseline_policy.onnx --task baseline)。
"""
import argparse
import time
from pathlib import Path

import mujoco
import mujoco.viewer
import numpy as np
import onnxruntime as ort

DECIMATION = 5              # 学習時と同じ制御周期(物理5ステップに1回だけpolicyを呼ぶ)
PHYSICS_TIMESTEP = 0.01     # cartpole.xmlの<option timestep="0.01"/>と一致させる
CTRL_RANGE = (-1.5, 1.5)    # cartpole.xmlのactuator ctrlrangeと一致させる
TARGET_STEP = 0.1
TARGET_RANGE = (-1.2, 1.2)  # CartTargetCommandCfg.target_rangeと一致させる


def get_joint_addrs(model: mujoco.MjModel, joint_name: str) -> tuple[int, int]:
    """関節名からqpos/qvel配列上のインデックスを取得する。"""
    jid = mujoco.mj_name2id(model, mujoco.mjtObj.mjOBJ_JOINT, joint_name)
    if jid < 0:
        raise ValueError(f"joint '{joint_name}' が見つかりません")
    return int(model.jnt_qposadr[jid]), int(model.jnt_dofadr[jid])


class CartpoleObservation:
    """観測ベクトルを組み立てるクラス。

    並び順・内容は、Notebook内のObservationTermCfgの定義と完全に一致させる必要がある。
    ここがズレると、ONNXは正常に動くがロボット(この場合はCartPole)は
    まったく見当違いの入力を渡されて暴走する、という一番気づきにくい不具合になる。
    """

    def __init__(self, model: mujoco.MjModel, task: str):
        self.task = task
        self.cart_qpos, self.cart_qvel = get_joint_addrs(model, "slider")
        self.hinge_qpos, self.hinge_qvel = get_joint_addrs(model, "hinge_1")
        self.target = 0.0  # tier3のみ使用。キー入力で外部から更新される

    def build(self, data: mujoco.MjData) -> np.ndarray:
        cart_pos = float(data.qpos[self.cart_qpos])
        theta = float(data.qpos[self.hinge_qpos])
        cart_vel = float(data.qvel[self.cart_qvel])
        pole_vel = float(data.qvel[self.hinge_qvel])

        if self.task == "tier3":
            # cart_pos, cos(theta), sin(theta), cart_vel, pole_vel, cart_target (6次元)
            obs = [cart_pos, np.cos(theta), np.sin(theta), cart_vel, pole_vel, self.target]
        else:
            # baseline / tier2a_domain_rand (5次元)。ドメインランダム化はイベントのみの変更なので観測形状は同じ
            obs = [cart_pos, np.cos(theta), np.sin(theta), cart_vel, pole_vel]
        return np.asarray(obs, dtype=np.float32)


def main():
    parser = argparse.ArgumentParser(description="CartPole ONNX policy inference (CPU-only, no mjlab)")
    parser.add_argument("--onnx", required=True, help="ONNXポリシーファイルのパス")
    parser.add_argument("--xml", default="cartpole.xml", help="MJCFファイルのパス")
    parser.add_argument(
        "--task",
        choices=["baseline", "tier3"],
        default="baseline",
        help="観測の組み立て方(次元)を選ぶ。tier2a_domain_randはbaselineと同じ観測形状なのでbaselineを指定する",
    )
    args = parser.parse_args()

    model = mujoco.MjModel.from_xml_path(args.xml)
    model.opt.timestep = PHYSICS_TIMESTEP
    data = mujoco.MjData(model)

    session = ort.InferenceSession(args.onnx, providers=["CPUExecutionProvider"])
    input_name = session.get_inputs()[0].name
    output_name = session.get_outputs()[0].name
    print(f"Loaded ONNX policy: input={session.get_inputs()[0].shape}, output={session.get_outputs()[0].shape}")

    obs_builder = CartpoleObservation(model, args.task)
    actuator_id = mujoco.mj_name2id(model, mujoco.mjtObj.mjOBJ_ACTUATOR, "slide")

    # swing-up初期状態(ポールを下向きに垂らした状態からスタート)
    data.qpos[obs_builder.hinge_qpos] = np.pi
    mujoco.mj_forward(model, data)

    def key_callback(keycode: int):
        """GLFWキーコード: 262=Right, 263=Left。tier3のときだけ目標カート位置を動かす。"""
        if args.task != "tier3":
            return
        if keycode == 262:
            obs_builder.target = min(obs_builder.target + TARGET_STEP, TARGET_RANGE[1])
            print(f"[key] 目標カート位置: {obs_builder.target:.2f}")
        elif keycode == 263:
            obs_builder.target = max(obs_builder.target - TARGET_STEP, TARGET_RANGE[0])
            print(f"[key] 目標カート位置: {obs_builder.target:.2f}")

    with mujoco.viewer.launch_passive(model, data, key_callback=key_callback) as viewer:
        print("ビューアを起動しました。ウィンドウを閉じると終了します。")
        while viewer.is_running():
            step_start = time.time()

            obs = obs_builder.build(data).reshape(1, -1)
            action = session.run([output_name], {input_name: obs})[0]
            ctrl = float(np.clip(action[0, 0], *CTRL_RANGE))

            # 学習時と同じdecimation(物理DECIMATIONステップに1回だけpolicyを呼ぶ)
            data.ctrl[actuator_id] = ctrl
            for _ in range(DECIMATION):
                mujoco.mj_step(model, data)

            viewer.sync()

            # 実時間に同期させる(ラフなタイムキーピング。ドリフトは許容する)
            elapsed = time.time() - step_start
            sleep_time = (DECIMATION * PHYSICS_TIMESTEP) - elapsed
            if sleep_time > 0:
                time.sleep(sleep_time)


if __name__ == "__main__":
    main()


---
## 参考:意図的に「不適切な」実装に戻して比較する

baselineの学習が完了したら、次の改変を1つずつ試して再学習し、WandBの学習曲線と実際のポリシーの挙動(この後のplayやローカル実行)がどう変わるかを確認することをおすすめします。

- **報酬を戻す**:`rewards`を、Rewardsの節で「現状の問題のある実装」として示した、状態や行動によらず常に+1を返す一様な報酬に戻して学習させる
- **観測を戻す**:`pole_angle`の観測を、cos/sinではなくObservationの節で示した生角度θそのものに戻して学習させる
- **entropy_coefを変える**:`cartpole_ppo_runner_cfg(entropy_coef=...)`を極端に低い値(0.0)や高い値(0.1)に変えて学習させる

それぞれbaselineとは別のrun名(例:`ablation_reward`、`ablation_obs`、`ablation_entropy_low`)で学習させ、WandB上で並べて比較してみてください。「なぜこの実装が必要なのか」を、うまくいかない場合と比較することで体感できるはずです。

---
# 2. ドメインランダマイゼーション

mjlabは`mjlab.envs.mdp.dr`モジュールに、質量・慣性・関節減衰・摩擦などを型付きで安全にランダム化する関数群を用意しています(`dr.geom_friction`、`dr.body_mass`、`dr.pseudo_inertia`など)。ここではポールの質量・慣性(`dr.pseudo_inertia`)と、カート・ポール双方の関節減衰(`dr.joint_damping`)をエピソードごとにランダム化します。

> 質量だけを変えて慣性はそのままにする(`dr.body_mass`単体)のは物理的に不整合になりやすいため、公式ドキュメントでも質量と慣性を同時にスケールする`dr.pseudo_inertia`(alpha_range)の使用が推奨されています。

**見どころ**:baselineより学習がやや遅くなる、あるいは最終性能が若干下がることが多いですが、初期条件のばらつきに対して頑健な(=様々な質量・減衰でも倒れにくい)方策になっているかをこの後のplayで確認してください。


In [ ]:
from mjlab.envs.mdp import dr

def make_domain_rand_cfg(swing_up: bool = True, num_envs: int = NUM_ENVS):
    cfg = cartpole_env_cfg(swing_up=swing_up, num_envs=num_envs)
    pole_cfg = SceneEntityCfg("cartpole", body_names=["pole_1"])
    hinge_j_cfg = SceneEntityCfg("cartpole", joint_names=["hinge_1"])
    slider_j_cfg = SceneEntityCfg("cartpole", joint_names=["slider"])

    # baselineのeventsに、ドメインランダム化の項を追加する
    cfg.events = dict(cfg.events)
    cfg.events["dr_pole_mass_inertia"] = EventTermCfg(
        mode="reset",
        func=dr.pseudo_inertia,
        params={"asset_cfg": pole_cfg, "alpha_range": (-0.3, 0.3)},  # 質量・慣性を e^(2*alpha) でスケール
    )
    cfg.events["dr_hinge_damping"] = EventTermCfg(
        mode="reset",
        func=dr.joint_damping,
        params={"asset_cfg": hinge_j_cfg, "ranges": (0.5, 3.0), "operation": "scale"},
    )
    cfg.events["dr_slider_damping"] = EventTermCfg(
        mode="reset",
        func=dr.joint_damping,
        params={"asset_cfg": slider_j_cfg, "ranges": (0.5, 2.0), "operation": "scale"},
    )
    return cfg

tier2a_env_cfg = make_domain_rand_cfg()
tier2a_rl_cfg = cartpole_ppo_runner_cfg(entropy_coef=0.005)

log_dir_tier2a = run_training(tier2a_env_cfg, tier2a_rl_cfg, run_name="tier2a_domain_rand", max_iters=MAX_ITERS)


## ローカルでの確認

`play_local.py`・`infer_local.py`はどちらも1章で作成したものをそのまま使います。チェックポイントとONNXファイルだけ、このTier2a用に差し替えます。

In [ ]:
tier2a_ckpts = sorted(glob.glob(f"{log_dir_tier2a}/model_*.pt"))
checkpoint_path_tier2a = tier2a_ckpts[-1]
print("使用するチェックポイント:", checkpoint_path_tier2a)

files.download(checkpoint_path_tier2a)

```bash
uv run play_local.py --checkpoint model_499.pt --task tier2a_domain_rand
```

In [ ]:
export_to_onnx(
    env_cfg=make_domain_rand_cfg(swing_up=True, num_envs=1),
    rl_cfg=cartpole_ppo_runner_cfg(entropy_coef=0.005),
    checkpoint_path=tier2a_ckpts[-1],
    onnx_path="tier2a_policy.onnx",
)
files.download("tier2a_policy.onnx")

```bash
uv run infer_local.py --onnx tier2a_policy.onnx --task baseline
```

tier2a_domain_randはドメインランダム化イベントを追加しただけで観測の形状はbaselineと同じなので、`--task`は`baseline`のまま指定します。

---
# 3. コマンド入力の実装

Balanceタスクを、**カートの目標位置を外部から指定できるコマンド条件付きタスク**に拡張します。今までは「中央(0)で立てる」だけでしたが、これからは「指定した位置で立てる」に一般化します。学習時は目標をランダムに、play時には外部入力(スライダー操作)で与えられる設計です。

## 実装のポイント

- `CommandTerm`を継承したカスタムコマンド項`CartTargetCommand`を定義し、目標カート位置をエピソード中一定間隔でランダムにリサンプルする
- 報酬の`centered`項(baselineでは0を中心にtoleranceを取っていた部分)を、**目標位置を中心にする**よう1行だけ変更する。他の項(upright/small_ctrl/small_vel)は無改造
- 観測に`generated_commands`(mjlabの組み込み観測関数)で現在の目標値を追加する

`_update_command(env_ids)`は、mjlabの現行バージョンでは`env_ids`引数(全環境更新時は`None`、リセット直後はリセットされた環境のid)を受け取る必要があります。今回の目標値は一定間隔でのみ切り替わり毎ステップの補間はしないため、このメソッドは何もしなくて構いません。


In [ ]:
from mjlab.managers.command_manager import CommandTerm, CommandTermCfg
from mjlab.envs.mdp import generated_commands

class CartTargetCommand(CommandTerm):
    """カートの目標スライダー位置を生成するコマンド項。"""

    def __init__(self, cfg: "CartTargetCommandCfg", env):
        super().__init__(cfg, env)
        self._target = torch.zeros(env.num_envs, device=env.device)
        self._external_override = None  # play時にここへ外部入力を注入する
        # スライダー関節のインデックスを事前に引いておく(_update_metricsで使う)
        self._cart_joint_idx = list(env.scene["cartpole"].joint_names).index("slider")

    @property
    def command(self) -> torch.Tensor:
        return self._target.unsqueeze(-1)  # [num_envs, 1]

    def _resample_command(self, env_ids: torch.Tensor):
        if self._external_override is not None:
            self._target[env_ids] = self._external_override[env_ids]
        else:
            self._target[env_ids] = torch.empty(
                len(env_ids), device=self.device
            ).uniform_(*self.cfg.target_range)

    def _update_command(self, env_ids):
        # 一定間隔でのみ切り替える設計のため、毎ステップの更新は不要
        if self._external_override is not None:
            self._target[:] = self._external_override

    def _update_metrics(self):
        cart = self._env.scene["cartpole"]
        cart_pos = cart.data.joint_pos[:, self._cart_joint_idx]
        self.metrics["cart_target_error"] = (self._target - cart_pos).abs()

    def set_external_target(self, target):
        self._external_override = target


class CartTargetCommandCfg(CommandTermCfg):
    entity_name: str = "cartpole"
    resampling_time_range: tuple = (3.0, 6.0)
    target_range: tuple = (-1.2, 1.2)  # スライダーの可動範囲(-1.8, 1.8)より少し内側

    def build(self, env) -> CartTargetCommand:
        return CartTargetCommand(self, env)


In [ ]:
def cartpole_command_reward(env, cart_cfg: SceneEntityCfg, hinge_cfg: SceneEntityCfg,
                             command_name: str = "cart_target") -> torch.Tensor:
    """baselineのsmooth_rewardと構造は同じ。centered項だけ、0ではなくコマンドの目標を中心にする。"""
    cart: "Entity" = env.scene[cart_cfg.name]
    angle = cart.data.joint_pos[:, hinge_cfg.joint_ids]
    cos_angle = torch.cos(angle).squeeze(-1)
    cart_pos = cart.data.joint_pos[:, cart_cfg.joint_ids].squeeze(-1)
    pole_vel = cart.data.joint_vel[:, hinge_cfg.joint_ids].squeeze(-1)
    ctrl = env.action_manager.action.squeeze(-1)
    target = env.command_manager.get_command(command_name).squeeze(-1)

    upright = (cos_angle + 1.0) / 2.0
    centered = (1.0 + _tolerance(cart_pos - target, margin=1.0)) / 2.0  # ← baselineとの唯一の違い
    small_ctrl = (4.0 + _tolerance(ctrl, margin=1.0)) / 5.0
    small_vel = (1.0 + _tolerance(pole_vel, margin=5.0)) / 2.0
    return upright * centered * small_ctrl * small_vel

def make_command_conditioned_cfg(swing_up: bool = True, num_envs: int = NUM_ENVS):
    cfg = cartpole_env_cfg(swing_up=swing_up, num_envs=num_envs)
    cart_cfg = SceneEntityCfg("cartpole", joint_names=("slider",))
    hinge_cfg = SceneEntityCfg("cartpole", joint_names=("hinge_1",))

    cfg.commands = {
        "cart_target": CartTargetCommandCfg(),
    }

    actor_terms = {
        "cart_pos": ObservationTermCfg(func=joint_pos_rel, params={"asset_cfg": cart_cfg}),
        "pole_angle": ObservationTermCfg(func=pole_angle_cos_sin, params={"asset_cfg": hinge_cfg}),
        "cart_vel": ObservationTermCfg(func=joint_vel_rel, params={"asset_cfg": cart_cfg}),
        "pole_vel": ObservationTermCfg(func=joint_vel_rel, params={"asset_cfg": hinge_cfg}),
        "cart_target": ObservationTermCfg(func=generated_commands, params={"command_name": "cart_target"}),
    }
    cfg.observations = {
        "actor": ObservationGroupCfg(actor_terms),
        "critic": ObservationGroupCfg({**actor_terms}),
    }

    cfg.rewards = {
        "command_reward": RewardTermCfg(
            func=cartpole_command_reward, weight=1.0,
            params={"cart_cfg": cart_cfg, "hinge_cfg": hinge_cfg, "command_name": "cart_target"},
        ),
    }
    return cfg

tier3_env_cfg = make_command_conditioned_cfg()
tier3_rl_cfg = cartpole_ppo_runner_cfg(entropy_coef=0.005)

log_dir_tier3 = run_training(tier3_env_cfg, tier3_rl_cfg, run_name="tier3_command_conditioned", max_iters=MAX_ITERS)


## ローカルでの確認

同じく`play_local.py`・`infer_local.py`は1章のものをそのまま使い回します。

In [ ]:
tier3_ckpts = sorted(glob.glob(f"{log_dir_tier3}/model_*.pt"))
checkpoint_path_tier3 = tier3_ckpts[-1]
print("使用するチェックポイント:", checkpoint_path_tier3)

files.download(checkpoint_path_tier3)

```bash
uv run play_local.py --checkpoint model_499.pt --task tier3
```

← / →キーで目標カート位置を操作できます。

In [ ]:
export_to_onnx(
    env_cfg=make_command_conditioned_cfg(swing_up=True, num_envs=1),
    rl_cfg=cartpole_ppo_runner_cfg(entropy_coef=0.005),
    checkpoint_path=tier3_ckpts[-1],
    onnx_path="tier3_policy.onnx",
)
files.download("tier3_policy.onnx")

```bash
uv run infer_local.py --onnx tier3_policy.onnx --task tier3
```

---
# 4. 結果の比較と学習済みモデルの実行

## WandBでの比較

3つの実験がすべて同じWandBプロジェクト・実験名(`cartpole_handson`)の異なるrun(`baseline`, `tier2a_domain_rand`, `tier3_command_conditioned`)として記録されています。WandBのプロジェクト画面で3つのrunを選択し、`Episode Reward`を1つのグラフに重ねて表示すると、それぞれの改変が学習曲線にどう影響したかを一望できます。


In [ ]:
import wandb

try:
    if wandb.run is not None:
        print("最新runのWandB URL:", wandb.run.url)
        print("プロジェクトのrun一覧はこのURLの上位階層(プロジェクトトップ)から確認できます。")
    else:
        print("wandb.run が見つかりません。学習セルの実行後にもう一度実行してください。")
except Exception as e:
    print("WandBの状態を取得できませんでした:", e)


## 学習済みモデルの実行(Colab上での動画デモ)

学習済みモデルをColab上でそのまま動かし、`mediapy`でオフスクリーン動画として確認します。ローカル環境は不要です。セル内の`DEMO_TASK`を`"baseline"`か`"tier3"`に切り替えることで、baseline(スイングアップ後にその場でバランスを取る様子)とTier3(外部から目標カート位置を変えながら追従する様子)のどちらのデモも同じセルで確認できます。

In [ ]:
import glob
import numpy as np
import mediapy as media
from rsl_rl.models.mlp_model import MLPModel

DEMO_TASK = "tier3"  # "baseline" に変えるとbaselineのデモになります

if DEMO_TASK == "baseline":
    cfg_fn, log_dir_demo = cartpole_env_cfg, log_dir_baseline
elif DEMO_TASK == "tier3":
    cfg_fn, log_dir_demo = make_command_conditioned_cfg, log_dir_tier3
else:
    raise ValueError(f"Unknown DEMO_TASK: {DEMO_TASK}")

ckpts = sorted(glob.glob(f"{log_dir_demo}/model_*.pt"))
assert ckpts, f"チェックポイントが見つかりません。{DEMO_TASK}の学習セルを先に実行してください。"
checkpoint_path = ckpts[-1]
print(f"[{DEMO_TASK}] 使用するチェックポイント:", checkpoint_path)

play_cfg = cfg_fn(swing_up=True, num_envs=1)
play_cfg.episode_length_s = 9999.0
play_env = ManagerBasedRlEnv(cfg=play_cfg, device=device)

obs_dim = play_env.observation_space.spaces["actor"].shape[-1]
act_dim = play_env.action_space.shape[-1]

dummy_obs = {"actor": torch.zeros(1, obs_dim, device=device)}
actor = MLPModel(
    obs=dummy_obs, obs_groups={"actor": ["actor"]}, obs_set="actor",
    output_dim=act_dim, hidden_dims=(64, 64), activation="elu",
    distribution_cfg={
        "class_name": "rsl_rl.modules.distribution.GaussianDistribution",
        "init_std": 1.0, "std_type": "scalar",
    },
).to(device)

checkpoint = torch.load(checkpoint_path, map_location=device)
actor.load_state_dict(checkpoint["actor_state_dict"])
actor.eval()

# Tier3のみ目標カート位置のコマンドを持つ。baselineの場合はコマンド操作を一切行わない
command_term = play_env.command_manager.get_term("cart_target") if DEMO_TASK == "tier3" else None

frames = []
obs_dict, _ = play_env.reset()
n_steps = 600
for step in range(n_steps):
    if command_term is not None:
        # 目標カート位置をゆっくり左右に振ってみる(実運用でのスライダー操作を模擬)
        target = 1.0 * np.sin(2 * np.pi * step / 300)
        command_term.set_external_target(torch.full((1,), target, device=device))

    with torch.no_grad():
        action = actor(obs_dict)
    obs_dict, reward, terminated, truncated, info = play_env.step(action)

    frame = play_env.render()
    if frame is not None:
        frames.append(frame)

    if terminated.any() or truncated.any():
        obs_dict, _ = play_env.reset()

print(f"{len(frames)} フレームを収集しました")
if frames:
    fps = int(1 / (play_cfg.sim.mujoco.timestep * play_env.cfg.decimation))
    media.show_video(frames, fps=fps)
else:
    print("render()がフレームを返しませんでした。mjlabのバージョンによってrender APIが異なる場合があるため、"
          "mjlab.envs.ManagerBasedRlEnv のドキュメントで最新のオフスクリーン描画方法を確認してください。")

## ローカルでの対話的な確認

`play_local.py`・`infer_local.py`を使ったローカルでの対話的な確認方法(環境準備・実行コマンド・操作方法)は1章「学習済みbaselineポリシーを動かす」にまとめてあります。Tier2a・Tier3用のチェックポイント/ONNXファイルのダウンロードと実行コマンドは、それぞれ2章・3章の「ローカルでの確認」セルで行っています。

---
# 5. ONNXエクスポートとCPUのみでのローカル実行

前章の`play_local.py`は、ローカル側にも**mjlab一式(PyTorch・mujoco-warp・GPU)**が必要でした。ここでは [pollen-robotics/microduck_rl](https://github.com/pollen-robotics/microduck_rl) の `export.py` / `infer_policy.py` と同じ考え方——**学習はGPU上のmjlabで行い、デプロイはCPU上のONNX + 素のmujocoで行う**——を、CartPoleの規模に合わせて最小構成で再現します。

エクスポートしたポリシーは、mjlabもPyTorchも入っていない、`mujoco`と`onnxruntime`だけのマシン(古いノートPCや将来的には実機のオンボードコンピュータ)でも動かせるようになります。


## 各Tierのローカル実行まとめ

`export_to_onnx`関数と`infer_local.py`は1章で導入済みです。baseline/Tier2a/Tier3それぞれのONNX化とローカル実行は、対応する章(1章・2章・3章)の「ローカルでの確認」セルですでに行っています。

| Tier | ONNXファイル | 生成した章 | 実行コマンド |
|---|---|---|---|
| baseline | `baseline_policy.onnx` | 1章 | `uv run infer_local.py --onnx baseline_policy.onnx --task baseline` |
| tier2a_domain_rand | `tier2a_policy.onnx` | 2章 | `uv run infer_local.py --onnx tier2a_policy.onnx --task baseline` |
| tier3_command_conditioned | `tier3_policy.onnx` | 3章 | `uv run infer_local.py --onnx tier3_policy.onnx --task tier3` |

---
## まとめ

| # | 実験 | この中で自分で実装したもの | 学んだこと |
|---|---|---|---|
| 1 | baseline | Observation(cos/sin) / Actions / Events / Rewards(密な報酬) / Terminations | 「問題のある実装」から出発し、ManagerBasedRlEnvCfgの5要素を1つずつ自分の手で組み立てる |
| 2 | tier2a_domain_rand | Events(ドメインランダム化の追加) | sim-to-real、初期条件のばらつきに対する頑健性 |
| 3 | tier3_command_conditioned | Command / Observation / Rewards の拡張 | ゴール条件付き方策への一般化 |

baselineは特に、Observationは生角度→cos/sin、Rewardsは一様な生存ボーナス→状態に応じた密な報酬、という「問題のある実装を自分で直す」演習になっていました。Actions/Events/Terminationsは最初から正しい設計でしたが、同じ形式(問題提起→TODO)で実装することで、5つの要素すべてが同じ`ManagerBasedRlEnvCfg`という枠組みの中の対等な部品であることを体感できたはずです。実装例そのものはこのNotebookには載せていないので、詰まったときはCraftドキュメントの該当セクションを確認してください。

理論的な背景(PPOの仕組み、On-policy/Off-policy、分散環境での学習など)はCraftドキュメントにまとめてあるので、実装で疑問に思った点があればそちらも参照してください。

mjlabはベータ版で開発が活発なため、エラーが出た場合はまず[公式ドキュメント](https://mujocolab.github.io/mjlab/)と[GitHubリポジトリ](https://github.com/mujocolab/mjlab)の最新情報を確認してください。
